# SBERT — Unsupervised STS Evaluation

Evaluate the **NLI-trained SBERT-base model** on the unsupervised STS tasks from the original SBERT paper.

Pipeline:

```text
NLI-trained SBERT
      ↓
BERT + MEAN pooling
      ↓
sentence embeddings
      ↓
cosine similarity
      ↓
Spearman correlation with gold scores
      ↓
STS12, STS13, STS14, STS15, STS16, STS-B, SICK-R
```

The NLI classifier is not used during STS evaluation.


## 1. Imports and Configuration

In [43]:
import math
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModel
from scipy.stats import spearmanr

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


PyTorch: 2.11.0+cu128
CUDA available: True
GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU


In [44]:
DATA_DIR = Path("..") / "data"
STS_DIR = DATA_DIR / "STS"

MODEL_DIR = Path("outputs") / "sbert_nli_base"


# ------------------------------------------------------------
# Evaluation configuration
# ------------------------------------------------------------

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

MAX_LENGTH = 128
EVAL_BATCH_SIZE = 32


print("STS directory:")
print(STS_DIR.resolve())

print("\nModel directory:")
print(MODEL_DIR.resolve())

print("\nDevice:")
print(DEVICE)

STS directory:
F:\SEM-7\NLP\Project\data\STS

Model directory:
F:\SEM-7\NLP\Project\notebooks\outputs\sbert_nli_base

Device:
cuda


## 2. Verify the Trained Model

The trained model from `01_SBERT_NLI_Training.ipynb` should be inside:

```text
outputs/sbert_nli_base/
```


In [45]:
print("Model directory exists:", MODEL_DIR.exists())

if MODEL_DIR.exists():
    for path in sorted(MODEL_DIR.iterdir()):
        print(" -", path.name)


Model directory exists: True
 - model
 - tokenizer
 - training_history.csv


## 3. Mean Pooling

In [46]:
def mean_pooling(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    summed = torch.sum(last_hidden_state * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / counts


## 4. Load the Trained BERT Encoder

For STS we discard the NLI classifier and keep only:

```text
trained BERT → MEAN pooling → 768-D sentence embedding
```

This assumes the training notebook saved the encoder with `save_pretrained()`.

If your saved model uses a different format, we will adapt this cell to that exact format.


In [47]:
# ============================================================
# 4. LOAD THE TRAINED SBERT ENCODER
# ============================================================

MODEL_NAME = "bert-base-uncased"

MODEL_PATH = MODEL_DIR / "model" / "sbert_nli_base.pt"
TOKENIZER_PATH = MODEL_DIR / "tokenizer"


print("Model checkpoint:")
print(MODEL_PATH.resolve())

print("\nTokenizer path:")
print(TOKENIZER_PATH.resolve())


# ------------------------------------------------------------
# Verify files
# ------------------------------------------------------------

if not MODEL_PATH.exists():
    raise FileNotFoundError(
        f"Model checkpoint not found:\n{MODEL_PATH.resolve()}"
    )

if not TOKENIZER_PATH.exists():
    raise FileNotFoundError(
        f"Tokenizer not found:\n{TOKENIZER_PATH.resolve()}"
    )


# ------------------------------------------------------------
# Load tokenizer
# ------------------------------------------------------------

tokenizer = AutoTokenizer.from_pretrained(
    TOKENIZER_PATH
)

print("\nTokenizer loaded successfully.")


# ------------------------------------------------------------
# Create the original BERT architecture
# ------------------------------------------------------------

encoder = AutoModel.from_pretrained(
    MODEL_NAME
)


# ------------------------------------------------------------
# Load checkpoint
# ------------------------------------------------------------

checkpoint = torch.load(
    MODEL_PATH,
    map_location="cpu"
)

print("\nCheckpoint loaded.")
print("Checkpoint keys:")
print(list(checkpoint.keys()))


# ------------------------------------------------------------
# Extract actual model weights
# ------------------------------------------------------------

model_state_dict = checkpoint["model_state_dict"]


# ------------------------------------------------------------
# Extract only BERT weights
# ------------------------------------------------------------

bert_state_dict = {
    key[len("bert."):]: value
    for key, value in model_state_dict.items()
    if key.startswith("bert.")
}


print("\nNumber of BERT parameters:", len(bert_state_dict))


# ------------------------------------------------------------
# Load trained BERT weights
# ------------------------------------------------------------

missing_keys, unexpected_keys = encoder.load_state_dict(
    bert_state_dict,
    strict=False
)


# ------------------------------------------------------------
# Move model to device
# ------------------------------------------------------------

encoder = encoder.to(DEVICE)
encoder.eval()


# ------------------------------------------------------------
# Verification
# ------------------------------------------------------------

print("\nEncoder loaded successfully.")
print("Hidden size:", encoder.config.hidden_size)
print("Device:", DEVICE)

print("\nMissing keys:", len(missing_keys))
print("Unexpected keys:", len(unexpected_keys))

if missing_keys:
    print("\nMissing keys:")
    print(missing_keys)

if unexpected_keys:
    print("\nUnexpected keys:")
    print(unexpected_keys)

Model checkpoint:
F:\SEM-7\NLP\Project\notebooks\outputs\sbert_nli_base\model\sbert_nli_base.pt

Tokenizer path:
F:\SEM-7\NLP\Project\notebooks\outputs\sbert_nli_base\tokenizer

Tokenizer loaded successfully.

Checkpoint loaded.
Checkpoint keys:
['model_state_dict', 'model_name', 'pooling', 'num_labels', 'max_length', 'seed', 'epochs', 'learning_rate', 'effective_batch_size', 'physical_batch_size', 'gradient_accumulation_steps']

Number of BERT parameters: 199

Encoder loaded successfully.
Hidden size: 768
Device: cuda

Missing keys: 0
Unexpected keys: 0


## 5. Sentence Encoder

In [48]:
@torch.no_grad()
def encode_sentences(sentences, batch_size=EVAL_BATCH_SIZE):
    all_embeddings = []

    for start in tqdm(
        range(0, len(sentences), batch_size),
        desc="Encoding"
    ):
        batch = sentences[start:start + batch_size]

        inputs = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt"
        )

        inputs = {
            key: value.to(DEVICE)
            for key, value in inputs.items()
        }

        outputs = encoder(**inputs)

        embeddings = mean_pooling(
            outputs.last_hidden_state,
            inputs["attention_mask"]
        )

        all_embeddings.append(embeddings.cpu())

    return torch.cat(all_embeddings, dim=0)


## 6. Encoder Sanity Check

In [49]:
test_sentences = [
    "A man is eating food.",
    "A person is having a meal."
]

test_embeddings = encode_sentences(test_sentences)

print("Embedding shape:", test_embeddings.shape)
print("Expected: (2, 768)")


Encoding: 100%|██████████| 1/1 [00:00<00:00, 83.96it/s]

Embedding shape: torch.Size([2, 768])
Expected: (2, 768)


In [50]:
u = test_embeddings[0].numpy()
v = test_embeddings[1].numpy()

cosine = np.dot(u, v) / (
    np.linalg.norm(u) * np.linalg.norm(v) + 1e-12
)

print("Cosine similarity:", cosine)


Cosine similarity: 0.8563448044205629


## 7. STS Evaluation Function

For each sentence pair:

1. Encode sentence 1.
2. Encode sentence 2.
3. Calculate cosine similarity.
4. Calculate Spearman correlation against the human gold score.

The paper reports `Spearman × 100`.


In [51]:
def evaluate_sts_dataframe(df, name):
    sentences1 = df["sentence1"].astype(str).tolist()
    sentences2 = df["sentence2"].astype(str).tolist()
    gold = df["score"].astype(float).to_numpy()

    print(f"\n{name}")
    print("=" * len(name))
    print("Pairs:", len(df))

    emb1 = encode_sentences(sentences1).numpy()
    emb2 = encode_sentences(sentences2).numpy()

    numerator = np.sum(emb1 * emb2, axis=1)
    denominator = (
        np.linalg.norm(emb1, axis=1) *
        np.linalg.norm(emb2, axis=1)
    )

    predicted = numerator / (denominator + 1e-12)

    rho = spearmanr(predicted, gold).statistic
    score = rho * 100

    print(f"Spearman rho: {rho:.6f}")
    print(f"Paper-style score: {score:.2f}")

    return {
        "dataset": name,
        "score": score,
        "spearman": rho,
        "n_pairs": len(df)
    }


## 8. Inspect the Downloaded STS Data

Run this before writing the format-specific loaders. The STS12–STS16 and SICK-R datasets use different raw formats, so we inspect the actual files we downloaded rather than guessing their filenames.


In [52]:
def inspect_directory(root):
    print("Directory:", root.resolve())

    if not root.exists():
        print("Directory does not exist.")
        return

    for path in sorted(root.rglob("*")):
        if path.is_file():
            print(" -", path.relative_to(root))


for dataset_name in [
    "STS12", "STS13", "STS14", "STS15", "STS16", "STS-B", "SICK-R"
]:
    print("\n" + "=" * 70)
    print(dataset_name)
    print("=" * 70)
    inspect_directory(STS_DIR / dataset_name)



STS12
Directory: F:\SEM-7\NLP\Project\data\STS\STS12
 - dataset_dict.json
 - test\data-00000-of-00001.arrow
 - test\dataset_info.json
 - test\state.json
 - train\data-00000-of-00001.arrow
 - train\dataset_info.json
 - train\state.json

STS13
Directory: F:\SEM-7\NLP\Project\data\STS\STS13
 - dataset_dict.json
 - test\data-00000-of-00001.arrow
 - test\dataset_info.json
 - test\state.json

STS14
Directory: F:\SEM-7\NLP\Project\data\STS\STS14
 - dataset_dict.json
 - test\data-00000-of-00001.arrow
 - test\dataset_info.json
 - test\state.json

STS15
Directory: F:\SEM-7\NLP\Project\data\STS\STS15
 - dataset_dict.json
 - test\data-00000-of-00001.arrow
 - test\dataset_info.json
 - test\state.json

STS16
Directory: F:\SEM-7\NLP\Project\data\STS\STS16
 - dataset_dict.json
 - test\data-00000-of-00001.arrow
 - test\dataset_info.json
 - test\state.json

STS-B
Directory: F:\SEM-7\NLP\Project\data\STS\STS-B
Directory does not exist.

SICK-R
Directory: F:\SEM-7\NLP\Project\data\STS\SICK-R
 - dataset_d

## 9. STS-B

STS-B is the easiest dataset to normalize.

Expected normalized format:

```text
sentence1 | sentence2 | score
```

We use the **test split** for the Table 1 comparison.


In [61]:
# ============================================================
# 9. LOAD ALL STS DATASETS
# ============================================================

from datasets import load_from_disk


STS_DATASETS = {
    "STS12": STS_DIR / "STS12",
    "STS13": STS_DIR / "STS13",
    "STS14": STS_DIR / "STS14",
    "STS15": STS_DIR / "STS15",
    "STS16": STS_DIR / "STS16",
    "STS-B": STS_DIR / "STSb",
    "SICK-R": STS_DIR / "SICK-R",
}


sts_datasets = {}

for name, path in STS_DATASETS.items():

    print("\n" + "=" * 70)
    print(name)
    print("=" * 70)

    print("Path:", path.resolve())

    if not path.exists():
        raise FileNotFoundError(
            f"{name} not found:\n{path.resolve()}"
        )

    dataset = load_from_disk(path)

    sts_datasets[name] = dataset

    print(dataset)


STS12
Path: F:\SEM-7\NLP\Project\data\STS\STS12
DatasetDict({
    train: Dataset({
        features: ['split', 'sentence1', 'sentence2', 'score'],
        num_rows: 2234
    })
    test: Dataset({
        features: ['split', 'sentence1', 'sentence2', 'score'],
        num_rows: 3108
    })
})

STS13
Path: F:\SEM-7\NLP\Project\data\STS\STS13
DatasetDict({
    test: Dataset({
        features: ['split', 'sentence1', 'sentence2', 'score'],
        num_rows: 1500
    })
})

STS14
Path: F:\SEM-7\NLP\Project\data\STS\STS14
DatasetDict({
    test: Dataset({
        features: ['split', 'sentence1', 'sentence2', 'score'],
        num_rows: 3750
    })
})

STS15
Path: F:\SEM-7\NLP\Project\data\STS\STS15
DatasetDict({
    test: Dataset({
        features: ['split', 'sentence1', 'sentence2', 'score'],
        num_rows: 3000
    })
})

STS16
Path: F:\SEM-7\NLP\Project\data\STS\STS16
DatasetDict({
    test: Dataset({
        features: ['split', 'sentence1', 'sentence2', 'score'],
        num_rows: 

In [62]:
# ============================================================
# 10. NORMALIZE STS DATASETS
# ============================================================

def get_test_split(dataset):
    """
    Return the appropriate evaluation split.

    Preference:
        test
        validation
        train
    """

    if hasattr(dataset, "keys"):

        if "test" in dataset:
            return dataset["test"]

        if "validation" in dataset:
            return dataset["validation"]

        if "train" in dataset:
            return dataset["train"]

    return dataset


def normalize_sts_dataset(dataset, name):

    dataset = get_test_split(dataset)

    print(f"\n{name}")
    print("-" * len(name))

    print("Columns:", dataset.column_names)
    print("Examples:", len(dataset))

    columns = dataset.column_names

    # --------------------------------------------------------
    # Identify sentence 1
    # --------------------------------------------------------

    sentence1_candidates = [
        "sentence1",
        "sentence_1",
        "sent1",
        "sentence1",
        "text1",
    ]

    # --------------------------------------------------------
    # Identify sentence 2
    # --------------------------------------------------------

    sentence2_candidates = [
        "sentence2",
        "sentence_2",
        "sent2",
        "text2",
    ]

    # --------------------------------------------------------
    # Identify score
    # --------------------------------------------------------

    score_candidates = [
        "score",
        "similarity",
        "relatedness_score",
        "label",
    ]

    sentence1_col = next(
        (
            col
            for col in sentence1_candidates
            if col in columns
        ),
        None
    )

    sentence2_col = next(
        (
            col
            for col in sentence2_candidates
            if col in columns
        ),
        None
    )

    score_col = next(
        (
            col
            for col in score_candidates
            if col in columns
        ),
        None
    )

    if sentence1_col is None:
        raise ValueError(
            f"{name}: Could not find sentence1 column.\n"
            f"Available columns: {columns}"
        )

    if sentence2_col is None:
        raise ValueError(
            f"{name}: Could not find sentence2 column.\n"
            f"Available columns: {columns}"
        )

    if score_col is None:
        raise ValueError(
            f"{name}: Could not find score column.\n"
            f"Available columns: {columns}"
        )

    # --------------------------------------------------------
    # Convert to pandas
    # --------------------------------------------------------

    df = dataset.to_pandas()

    df = df[
        [
            sentence1_col,
            sentence2_col,
            score_col
        ]
    ].copy()

    df.columns = [
        "sentence1",
        "sentence2",
        "score"
    ]

    # --------------------------------------------------------
    # Convert score to numeric
    # --------------------------------------------------------

    df["score"] = pd.to_numeric(
        df["score"],
        errors="coerce"
    )

    # --------------------------------------------------------
    # Remove invalid rows
    # --------------------------------------------------------

    df = df.dropna(
        subset=[
            "sentence1",
            "sentence2",
            "score"
        ]
    ).reset_index(drop=True)

    print("Normalized columns:", df.columns.tolist())
    print("Final pairs:", len(df))

    return df

In [63]:
# ============================================================
# 11. NORMALIZE ALL DATASETS
# ============================================================

sts_data = {}

for name, dataset in sts_datasets.items():

    sts_data[name] = normalize_sts_dataset(
        dataset,
        name
    )


STS12
-----
Columns: ['split', 'sentence1', 'sentence2', 'score']
Examples: 3108
Normalized columns: ['sentence1', 'sentence2', 'score']
Final pairs: 3108

STS13
-----
Columns: ['split', 'sentence1', 'sentence2', 'score']
Examples: 1500
Normalized columns: ['sentence1', 'sentence2', 'score']
Final pairs: 1500

STS14
-----
Columns: ['split', 'sentence1', 'sentence2', 'score']
Examples: 3750
Normalized columns: ['sentence1', 'sentence2', 'score']
Final pairs: 3750

STS15
-----
Columns: ['split', 'sentence1', 'sentence2', 'score']
Examples: 3000
Normalized columns: ['sentence1', 'sentence2', 'score']
Final pairs: 3000

STS16
-----
Columns: ['split', 'sentence1', 'sentence2', 'score']
Examples: 1186
Normalized columns: ['sentence1', 'sentence2', 'score']
Final pairs: 1186

STS-B
-----
Columns: ['split', 'genre', 'dataset', 'year', 'sid', 'score', 'sentence1', 'sentence2']
Examples: 1379
Normalized columns: ['sentence1', 'sentence2', 'score']
Final pairs: 1379

SICK-R
------
Columns: ['sen

In [64]:
# ============================================================
# 12. VERIFY ALL STS DATASETS
# ============================================================

for name, df in sts_data.items():

    print("\n" + "=" * 70)
    print(name)
    print("=" * 70)

    print("Shape:", df.shape)

    print("Columns:", df.columns.tolist())

    print("Missing values:")
    print(df.isna().sum())

    print("\nFirst example:")

    print("Sentence 1:")
    print(df.iloc[0]["sentence1"])

    print("\nSentence 2:")
    print(df.iloc[0]["sentence2"])

    print("\nGold score:")
    print(df.iloc[0]["score"])


STS12
Shape: (3108, 3)
Columns: ['sentence1', 'sentence2', 'score']
Missing values:
sentence1    0
sentence2    0
score        0
dtype: int64

First example:
Sentence 1:
the problem likely will mean corrective changes before the shuttle fleet starts flying again .

Sentence 2:
he said the problem needs to be corrected before the space shuttle fleet is cleared to fly again .

Gold score:
4.4

STS13
Shape: (1500, 3)
Columns: ['sentence1', 'sentence2', 'score']
Missing values:
sentence1    0
sentence2    0
score        0
dtype: int64

First example:
Sentence 1:
there exist a number of different possible events that may happen in the future. in most cases , there is an agent involved who has to consider which of the possible events will or should occur. a salient _ entity which is deeply involved in the event may also be mentioned .

Sentence 2:
doing as one pleases or chooses ;

Gold score:
0.6

STS14
Shape: (3750, 3)
Columns: ['sentence1', 'sentence2', 'score']
Missing values:
sentence1

In [65]:
# ============================================================
# 13. EVALUATE ALL SEVEN STS DATASETS
# ============================================================

results = []

for name in [
    "STS12",
    "STS13",
    "STS14",
    "STS15",
    "STS16",
    "STS-B",
    "SICK-R"
]:

    result = evaluate_sts_dataframe(
        sts_data[name],
        name
    )

    results.append(result)


STS12
=====
Pairs: 3108


Encoding: 100%|██████████| 98/98 [00:05<00:00, 16.88it/s]


Spearman rho: 0.706645
Paper-style score: 70.66

STS13
=====
Pairs: 1500


Encoding: 100%|██████████| 47/47 [00:02<00:00, 21.97it/s]


Spearman rho: 0.727781
Paper-style score: 72.78

STS14
=====
Pairs: 3750


...

Spearman rho: 0.740588
Paper-style score: 74.06

STS-B
=====
Pairs: 1379


Encoding: 100%|██████████| 44/44 [00:02<00:00, 19.95it/s]


Spearman rho: 0.767793
Paper-style score: 76.78

SICK-R
Pairs: 9927


Encoding: 100%|██████████| 311/311 [00:12<00:00, 24.59it/s]


Spearman rho: 0.733630
Paper-style score: 73.36


## 10. Table 1 Reference Results

Original paper — `SBERT-NLI-base`:

| Dataset | Paper |
|---|---:|
| STS12 | 70.97 |
| STS13 | 76.53 |
| STS14 | 73.19 |
| STS15 | 79.09 |
| STS16 | 74.30 |
| STS-B | 77.03 |
| SICK-R | 72.91 |
| Average | 74.89 |

These are reference values for comparison, not values to reproduce by hard-coding.


In [66]:
# ============================================================
# 14. TABLE 1 COMPARISON
# ============================================================

paper_results = {
    "STS12": 70.97,
    "STS13": 76.53,
    "STS14": 73.19,
    "STS15": 79.09,
    "STS16": 74.30,
    "STS-B": 77.03,
    "SICK-R": 72.91,
}


our_df = pd.DataFrame(results)

paper_df = pd.DataFrame(
    list(paper_results.items()),
    columns=[
        "dataset",
        "paper_score"
    ]
)


comparison = paper_df.merge(
    our_df[
        [
            "dataset",
            "score"
        ]
    ],
    on="dataset",
    how="left"
)


comparison["difference"] = (
    comparison["score"]
    -
    comparison["paper_score"]
)


comparison

,dataset,paper_score,score,difference
0,STS12,70.97,70.664484,-0.305516
1,STS13,76.53,72.778056,-3.751944
2,STS14,73.19,70.662632,-2.527368
3,STS15,79.09,78.869953,-0.220047
4,STS16,74.30,74.058850,-0.241150
5,STS-B,77.03,76.779260,-0.250740
6,SICK-R,72.91,73.363016,0.453016


In [67]:
# ============================================================
# 15. AVERAGE
# ============================================================

if comparison["score"].notna().all():

    our_average = comparison["score"].mean()
    paper_average = comparison["paper_score"].mean()

    print(
        "Our average:",
        f"{our_average:.2f}"
    )

    print(
        "Paper average:",
        f"{paper_average:.2f}"
    )

    print(
        "Average difference:",
        f"{our_average - paper_average:.2f}"
    )

else:

    print(
        "Some datasets were not evaluated."
    )

Our average: 73.88
Paper average: 74.86
Average difference: -0.98


# Next Experiment

After Table 1 is complete, the next notebook will implement the paper's **STS-B supervised fine-tuning** experiment for Table 2:

```text
NLI-trained SBERT
      ↓
STS-B training set
      ↓
Regression objective
      ↓
STS-B test set
      ↓
Spearman
```
